# Atlas integration analysis

Phase 3a/3b review notebook for the merged lung atlas. Loads `output/atlas/data/lung_atlas.h5ad` (or runs `run_atlas_integration` on a subset), compares UMAP embeddings before and after Harmony, inspects integration metrics, and builds a CyteType vs `cell_type` confusion matrix excluding `unknown`.

See also: [`writeups/atlas/README.md`](../../writeups/atlas/README.md)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

from atlas_integration import AtlasIntegrationConfig, run_atlas_integration
from shared.repo import REPO_ROOT

ATLAS_DIR = REPO_ROOT / "output" / "atlas"
ATLAS_PATH = ATLAS_DIR / "data" / "lung_atlas.h5ad"
METADATA_PATH = ATLAS_DIR / "run_metadata.json"
UNKNOWN_LABEL = "unknown"
CYTETYPE_COL = "cytetype_annotation_leiden_atlas"

## Build or load atlas

Set `runIntegration = True` to rebuild on a subset (recommended for local prototyping). The server runner [`pipelines/run_atlas_integration.py`](../../pipelines/run_atlas_integration.py) handles the full 772-accession merge.

In [ ]:
runIntegration = False
prototypeAccessionLimit = 5

if runIntegration:
    datasetsPath = REPO_ROOT / "output" / "metadata" / "datasets.csv"
    prototypeCsv = ATLAS_DIR / "datasets_prototype.csv"
    ATLAS_DIR.mkdir(parents=True, exist_ok=True)
    pd.read_csv(datasetsPath).head(prototypeAccessionLimit).to_csv(prototypeCsv, index=False)

    cfg = AtlasIntegrationConfig(
        datasetsCsvPath=prototypeCsv,
        outputDir=ATLAS_DIR,
        figsDir=ATLAS_DIR / "figs",
        subsampleN=1000,
    )
    adata, result = run_atlas_integration(cfg)
else:
    adata = sc.read_h5ad(ATLAS_PATH)
    result = None
    if METADATA_PATH.is_file():
        result = json.loads(METADATA_PATH.read_text())

adata

## Phase 3a: UMAP with and without batch correction

In [ ]:
cfg = AtlasIntegrationConfig()

for colorBy in (cfg.batchKey, cfg.cellTypeKey):
    sc.pl.embedding(
        adata,
        basis=cfg.umapKeyUncorrected.replace("X_", ""),
        color=colorBy,
        title=f"{colorBy} (uncorrected)",
        show=False,
    )
    sc.pl.embedding(
        adata,
        basis=cfg.umapKeyCorrected.replace("X_", ""),
        color=colorBy,
        title=f"{colorBy} (Harmony)",
        show=False,
    )
plt.show()

## Phase 3b: integration metrics

In [ ]:
if result is not None:
    if hasattr(result, "model_dump"):
        metrics = {
            "mergeStats": result.mergeStats.model_dump(),
            "batchMixing": result.batchMixing.model_dump(),
            "clusterConservation": result.clusterConservation.model_dump(),
        }
    else:
        metrics = result

    pd.DataFrame(
        [
            metrics["batchMixing"],
            metrics["clusterConservation"],
        ],
        index=["batchMixing", "clusterConservation"],
    )
else:
    print(f"No metrics found at {METADATA_PATH}")

## CyteType confusion matrix (`cell_type` vs CyteType)

Run CyteType on the atlas first (requires `CYTETYPE_API_KEY`). This section assumes `adata.obs[CYTETYPE_COL]` exists and excludes `unknown` author labels from the reference comparison.

In [ ]:
if CYTETYPE_COL not in adata.obs:
    print(f"Column {CYTETYPE_COL!r} not found. Run CyteType on leiden_atlas clusters first.")
else:
    labeled = adata.obs[adata.obs["cell_type"] != UNKNOWN_LABEL].copy()
    confusion = pd.crosstab(labeled["cell_type"], labeled[CYTETYPE_COL])
    confusion.iloc[:10, :10]